# FASE 2: Preprocessing & Feature Engineering
## Prediksi PM2.5 di Cekungan Bandung Menggunakan LSTM dengan Interpretasi SHAP
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini melanjutkan hasil dari Eksplorasi Data (EDA) dengan fokus menyiapkan data agar siap dimasukkan ke dalam arsitektur *Machine Learning* (Random Forest, XGBoost) dan *Deep Learning* (LSTM, BiLSTM).

**Langkah-langkah utama di notebook ini:**
1. **Handling Missing Values:** Interpolasi nilai `NaN` pada PM2.5 menggunakan metode berbasis waktu.
2. **Feature Engineering:** Ekstraksi fitur waktu siklik (*Cyclical Encoding*), penanda musim, dan variabel Lag historis.
3. **Chronological Train-Validation-Test Split (70/10/20):** Membagi data secara berurutan waktu dengan tiga partisi eksplisit.
4. **Data Normalization:** Penskalaan fitur (MinMaxScaler) yang wajib dilakukan sebelum training model berbasis Neural Network.


## 1. Import Library


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print("[OK] Library loaded.")


## 2. Load Dataset Merged (Kontinu)
Kita menggunakan data `merged_pm25_era5v3.csv` yang berisi 1 tahun data berurutan (Nov 2022 - Nov 2023) dengan total 9.480 baris per jam.

> **Justifikasi Penggunaan Data 1 Tahun (Resolusi Per Jam)**
>
> Keputusan untuk menggunakan cakupan data 1 tahun dengan resolusi per jam didasarkan pada dua pertimbangan ilmiah utama:
>
> 1. **Kelengkapan Siklus Musiman:** Satu tahun penuh memastikan model dapat mempelajari kedua pola musim secara seimbang, yaitu musim **kemarau (Juni–September)** dan musim **hujan (Oktober–Mei)**. Penelitian Yin et al. (2021) dan Chen et al. (2024) menegaskan bahwa setidaknya satu siklus musiman tahunan penuh diperlukan agar model mampu menggeneralisasi variasi cuaca yang mempengaruhi konsentrasi polutan.
>
> 2. **Kecukupan Volume Data untuk Deep Learning:** Dengan resolusi per jam, 1 tahun data menghasilkan ±8.760 sampel, melampaui ambang batas *"rule of thumb"* minimum yang direkomendasikan oleh Goodfellow et al. (2016) untuk pelatihan arsitektur LSTM yang stabil, yaitu ≥5.000 sampel per kelas/target. Sebaliknya, resolusi harian (daily) hanya menghasilkan 365 baris — volume yang terbukti tidak cukup untuk konvergensi model *Deep Learning* dan rentan terhadap *overfitting* (Lim & Zohren, 2021).


In [ ]:
# Load data dari Google Drive
data_path = '/content/drive/MyDrive/KP_BRIN/data/raw/merged_pm25_era5v3.csv'
df = pd.read_csv(data_path)

# Rename kolom pertama menjadi datetime dan jadikan index
df.rename(columns={df.columns[0]: 'datetime'}, inplace=True)
df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

print(f"Shape Dataset  : {df.shape}")
print(f"Periode        : {df.index.min()} --> {df.index.max()}")
print(f"Total Missing PM2.5 : {df['pm25'].isnull().sum()} baris ({df['pm25'].isnull().mean()*100:.1f}%)")
df.head()


## 3. Handling Missing Values (Imputasi Time-Series)

Karena ini adalah data deret waktu (*time-series*), kita **DILARANG** menggunakan rata-rata global (*mean imputation*). Pengisian dengan rata-rata global akan merusak tren temporal dan memperkenalkan bias statistik yang serius — polutan 'tiba-tiba' kembali ke nilai rata-rata tanpa transisi yang realistis.

Kita menggunakan **Interpolasi Linear Berbasis Waktu** (`method='time'`) untuk menyambung secara mulus antara titik data terakhir sebelum sensor mati dan titik data pertama setelah sensor menyala kembali.

> **Justifikasi:** Metode ini valid karena konsentrasi PM2.5 bersifat kontinu secara fisika — tidak mungkin berubah secara tiba-tiba dari 0 µg/m³ ke 80 µg/m³ tanpa transisi. Metode interpolasi waktu telah menjadi standar dalam studi kualitas udara berbasis sensor *low-cost* (seperti yang diterapkan oleh Zheng et al., 2019 dan Kumar & Sahu, 2021).


In [ ]:
# Visualisasi contoh gap SEBELUM interpolasi (April-Mei 2023)
subset_gaps = df.loc['2023-04-10':'2023-05-10'].copy()

plt.figure(figsize=(15, 4))
plt.plot(subset_gaps.index, subset_gaps['pm25'], color='red',
         marker='.', linestyle='-', markersize=3, label='PM2.5 Asli (Ada Gap)')
plt.title("Kondisi Sebelum Interpolasi (April-Mei 2023) — Terlihat Gap (Data Hilang)")
plt.ylabel("PM2.5 (µg/m³)")
plt.xlabel("Waktu")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Lakukan interpolasi linear berbasis waktu
df['pm25'] = df['pm25'].interpolate(method='time')

# Verifikasi
print(f"Total Missing setelah interpolasi: {df['pm25'].isnull().sum()} baris")

# Visualisasi SETELAH interpolasi
subset_fixed = df.loc['2023-04-10':'2023-05-10'].copy()
plt.figure(figsize=(15, 4))
plt.plot(subset_fixed.index, subset_fixed['pm25'], color='green',
         marker='.', linestyle='-', markersize=3, label='PM2.5 Setelah Interpolasi')
plt.title("Kondisi Setelah Interpolasi (April-Mei 2023) — Gap Terisi Mulus")
plt.ylabel("PM2.5 (µg/m³)")
plt.xlabel("Waktu")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Feature Engineering

Menambahkan variabel/kolom baru yang akan sangat membantu model mempelajari pola musiman dan temporal dari data.

### 4.1 Ekstraksi Fitur Waktu (Temporal Features)


In [ ]:
# Ambil info jam, hari, dan bulan
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month

# Fitur Musim Biner (1=Kemarau, 0=Hujan)
# Kemarau di Bandung: Juni (6) - September (9)
df['is_dry_season'] = df['month'].apply(lambda x: 1 if x in [6, 7, 8, 9] else 0)

print("Fitur temporal berhasil ditambahkan:")
df[['hour', 'day_of_week', 'month', 'is_dry_season']].head()


### 4.2 Encoding Siklik (*Cyclical Encoding*)

Jam 23:00 dan jam 00:00 sebenarnya hanya berjarak 1 jam, tetapi secara numerik selisihnya 23. Hal yang sama berlaku untuk bulan: Desember (12) dan Januari (1) sebenarnya berdekatan, tetapi secara numerik selisihnya 11.

Jika kita membiarkan representasi numerik linear ini, algoritma akan salah menginterpretasikan jarak temporal. Oleh karena itu, fitur waktu harus dikodekan ke dalam bentuk gelombang **Sinus** dan **Cosinus** sehingga hubungan sikliknya terjaga:

$$\text{hour\_sin} = \sin\left(\frac{\text{jam} \times 2\pi}{24}\right), \quad \text{hour\_cos} = \cos\left(\frac{\text{jam} \times 2\pi}{24}\right)$$


In [ ]:
# Transformasi Siklik untuk Jam (Siklus 24 Jam)
df['hour_sin'] = np.sin(df['hour'] * (2. * np.pi / 24))
df['hour_cos'] = np.cos(df['hour'] * (2. * np.pi / 24))

# Transformasi Siklik untuk Bulan (Siklus 12 Bulan)
df['month_sin'] = np.sin((df['month'] - 1) * (2. * np.pi / 12))
df['month_cos'] = np.cos((df['month'] - 1) * (2. * np.pi / 12))

# Hapus kolom integer asli (tidak diperlukan lagi)
df.drop(columns=['hour', 'month'], inplace=True)

print("Encoding siklik selesai. Kolom baru: hour_sin, hour_cos, month_sin, month_cos")


### 4.3 Fitur Historis (*Lagged Features*)

Konsentrasi PM2.5 di atmosfer bersifat kumulatif: nilai saat ini sangat bergantung pada nilai jam-jam sebelumnya karena partikel tidak langsung hilang dari udara. Dua variabel lag yang paling penting ditambahkan:

- **`pm25_lag_1h`** → Nilai PM2.5 satu jam sebelumnya ($T-1$): Menangkap autokorelasi jangka pendek
- **`pm25_lag_24h`** → Nilai PM2.5 24 jam sebelumnya ($T-24$): Menangkap pola diurnal yang berulang setiap hari di jam yang sama


In [ ]:
# Lag 1 Jam (T-1)
df['pm25_lag_1h'] = df['pm25'].shift(1)

# Lag 24 Jam (T-24, pola diurnal)
df['pm25_lag_24h'] = df['pm25'].shift(24)

# Hapus 24 baris pertama yang menghasilkan NaN akibat shift
df.dropna(inplace=True)

print(f"Shape data setelah Feature Engineering: {df.shape}")
print(f"\nDaftar semua fitur ({len(df.columns)} kolom):")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")


## 5. Train - Validation - Test Split (70% / 10% / 20%)

Dalam pemodelan *Time-Series*, kita **DILARANG** membagi data secara acak (`train_test_split(shuffle=True)`). Pembagian acak menyebabkan **kebocoran data (*data leakage*)** — model secara tidak sengaja 'melihat' masa depan selama pelatihan, yang membuat evaluasi akurasi menjadi tidak valid.

Kita menggunakan pembagian **kronologis** (berurutan waktu) menjadi **tiga partisi eksplisit**:

| Partisi | Proporsi | Fungsi |
|---|---|---|
| **Training** | 70% | Melatih semua model (RF, XGBoost, LSTM, BiLSTM) |
| **Validation** | 10% | Tuning hyperparameter & Early Stopping LSTM |
| **Testing** | 20% | Evaluasi akurasi akhir yang murni dan adil |

> **Justifikasi Rasio 70/10/20:**
>
> Rasio ini merupakan standar yang banyak digunakan dalam penelitian prediksi *time-series* berbasis *Deep Learning* (Lim & Zohren, 2021; Yin et al., 2021). Alasan spesifik untuk setiap partisi:
>
> - **70% Training:** Memastikan model mendapatkan cukup data untuk mempelajari kedua siklus musiman (kemarau dan hujan). Dengan ~6.132 jam data latih, volume ini memenuhi kebutuhan minimum LSTM yang stabil.
> - **10% Validation (~876 jam ≈ 1 bulan):** Cukup representatif untuk mengevaluasi konvergensi model *epoch* per *epoch* (Early Stopping) dan juga sebagai dasar tuning hyperparameter Random Forest dan XGBoost secara adil.
> - **20% Testing (~1.752 jam ≈ 2,5 bulan):** Mengikuti prinsip bahwa data test harus cukup besar untuk mencakup variasi kondisi cuaca yang bermakna, sehingga evaluasi RMSE, MAE, dan R² dapat dianggap representatif secara statistik (Hyndman & Athanasopoulos, 2021).


In [ ]:
total = len(df)

# Hitung indeks batas
train_end = int(total * 0.70)
val_end   = int(total * 0.80)  # 70% + 10% = 80%

# Bagi secara kronologis
train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("=" * 65)
print(f"{'Partisi':<12} {'Baris':>8}  {'Dari':<22} {'Sampai':<22}")
print("=" * 65)
print(f"{'Training':<12} {len(train_df):>8,}  {str(train_df.index.min())[:19]:<22} {str(train_df.index.max())[:19]:<22}")
print(f"{'Validation':<12} {len(val_df):>8,}  {str(val_df.index.min())[:19]:<22} {str(val_df.index.max())[:19]:<22}")
print(f"{'Testing':<12} {len(test_df):>8,}  {str(test_df.index.min())[:19]:<22} {str(test_df.index.max())[:19]:<22}")
print("=" * 65)
print(f"Total baris  : {len(df):,}")
print(f"Rasio aktual : Train={len(train_df)/total*100:.1f}% / Val={len(val_df)/total*100:.1f}% / Test={len(test_df)/total*100:.1f}%")


In [ ]:
# Visualisasi pembagian data
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(train_df.index, train_df['pm25'], color='royalblue', linewidth=0.6,
        label=f'Training ({len(train_df):,} jam | 70%)')
ax.plot(val_df.index, val_df['pm25'], color='darkorange', linewidth=0.6,
        label=f'Validation ({len(val_df):,} jam | 10%)')
ax.plot(test_df.index, test_df['pm25'], color='forestgreen', linewidth=0.6,
        label=f'Testing ({len(test_df):,} jam | 20%)')

ax.axvline(x=train_df.index[-1], color='royalblue', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(x=val_df.index[-1], color='darkorange', linestyle='--', linewidth=1.5, alpha=0.7)

ax.set_title('Chronological Train/Validation/Test Split (70% / 10% / 20%)', fontsize=13, fontweight='bold')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_xlabel('Waktu')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Normalisasi (*Min-Max Scaling*)

Jaringan Saraf Tiruan (*Deep Learning* / LSTM) sangat sensitif terhadap perbedaan skala antar variabel. Misalnya, variabel `tekanan_hPa` memiliki rentang 890–901, sementara `tutupan_awan_01` hanya 0–1. Perbedaan skala ini menyebabkan model terdominasi oleh variabel berskala besar dan gradien menjadi tidak stabil (*exploding/vanishing gradient*).

Semua fitur ditekan ke rentang **[0, 1]** menggunakan `MinMaxScaler`.

> **Aturan Krusial (Anti Data Leakage):** Scaler di-*fit* **hanya menggunakan data Training**, lalu di-*transform* ke Validation dan Testing. Hal ini memastikan statistik minimum/maksimum dari data masa depan tidak bocor ke proses normalisasi data latih.


In [ ]:
target_col = 'pm25'
feature_cols = [c for c in df.columns if c != target_col]

# Inisialisasi dua scaler terpisah
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

# === FIT hanya pada Training ===
X_train_s = scaler_X.fit_transform(train_df[feature_cols])
y_train_s = scaler_y.fit_transform(train_df[[target_col]])

# === HANYA TRANSFORM (tanpa fit) pada Validation ===
X_val_s = scaler_X.transform(val_df[feature_cols])
y_val_s = scaler_y.transform(val_df[[target_col]])

# === HANYA TRANSFORM (tanpa fit) pada Testing ===
X_test_s = scaler_X.transform(test_df[feature_cols])
y_test_s = scaler_y.transform(test_df[[target_col]])

# Kembalikan ke DataFrame agar rapi
train_scaled = pd.DataFrame(X_train_s, columns=feature_cols, index=train_df.index)
train_scaled[target_col] = y_train_s

val_scaled = pd.DataFrame(X_val_s, columns=feature_cols, index=val_df.index)
val_scaled[target_col] = y_val_s

test_scaled = pd.DataFrame(X_test_s, columns=feature_cols, index=test_df.index)
test_scaled[target_col] = y_test_s

print("Contoh data Training setelah di-scaling (semua rentang 0 sampai 1):")
print(train_scaled.describe().round(3))


## 7. Simpan Data Siap Model

Data yang sudah bersih, direkayasa, dan diskalakan disimpan untuk digunakan oleh Notebook pemodelan (Notebook 03, 04, dan 05).


In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/KP_BRIN/data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Simpan tiga partisi data
train_scaled.to_csv(os.path.join(OUTPUT_DIR, 'train_scaled.csv'))
val_scaled.to_csv(os.path.join(OUTPUT_DIR, 'val_scaled.csv'))
test_scaled.to_csv(os.path.join(OUTPUT_DIR, 'test_scaled.csv'))

# Simpan Scaler (penting untuk membalikkan prediksi ke satuan ug/m3 asli)
joblib.dump(scaler_X, os.path.join(OUTPUT_DIR, 'scaler_X.pkl'))
joblib.dump(scaler_y, os.path.join(OUTPUT_DIR, 'scaler_y.pkl'))

print(f"[OK] Seluruh data dan Scaler berhasil disimpan di:")
print(f"     {OUTPUT_DIR}")
print(f"")
print(f"File yang disimpan:")
print(f"  -> train_scaled.csv  ({len(train_scaled):,} baris)")
print(f"  -> val_scaled.csv    ({len(val_scaled):,} baris)")
print(f"  -> test_scaled.csv   ({len(test_scaled):,} baris)")
print(f"  -> scaler_X.pkl")
print(f"  -> scaler_y.pkl")
